In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="SageBio/mva-hackathon-2026-data",
    filename="WGS_EX2312012_HGWCNDSX7.vcf.gz",
    repo_type="dataset",
    local_dir="./data",
)

print(f"File downloaded to: {file_path}")

In [ ]:
from huggingface_hub import list_repo_files

for f in list_repo_files("SageBio/mva-hackathon-2026-data", repo_type="dataset"):
    print(f)

In [ ]:
from huggingface_hub import hf_hub_download

for fn in [
    "Challenge_Clinical_Phenotype_1.docx",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi",
]:
    p = hf_hub_download(
        repo_id="SageBio/mva-hackathon-2026-data",
        filename=fn,
        repo_type="dataset",
        local_dir="./data",
    )
    print(p)

In [ ]:
import requests

GENES = ["BUB1B", "CEP57", "TRIP13"]
PAD = 20000

def gene_region(symbol):
    r = requests.get(
        f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{symbol}",
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    r.raise_for_status()
    d = r.json()
    return d["seq_region_name"], d["start"], d["end"]

regions = {}
for g in GENES:
    chrom, start, end = gene_region(g)
    regions[g] = (chrom, max(1, start - PAD), end + PAD)
    print(g, regions[g])

In [ ]:
import gzip

VCF = "./data/WGS_EX2312012_HGWCNDSX7.vcf.gz"

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("##INFO") or line.startswith("##FORMAT"):
            print(line.strip())
        elif line.startswith("#CHROM"):
            print(line.strip())
            break

In [ ]:
import gzip

by_chrom = {}
for g, (c, s, e) in regions.items():
    by_chrom.setdefault(c, []).append((g, s, e))

hits = []
with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        tab1 = line.index("\t")
        chrom = line[:tab1].removeprefix("chr")
        spans = by_chrom.get(chrom)
        if spans is None:
            continue
        tab2 = line.index("\t", tab1 + 1)
        pos = int(line[tab1 + 1:tab2])
        for g, s, e in spans:
            if s <= pos <= e:
                hits.append([g] + line.rstrip("\n").split("\t"))
                break

print(len(hits), "variants in panel regions")

In [ ]:
import pandas as pd

cols = ["gene", "chrom", "pos", "id", "ref", "alt", "qual", "filter", "info", "format", "sample"]
df = pd.DataFrame(hits, columns=cols[:len(hits[0])])
df["pos"] = df["pos"].astype(int)

print(df["gene"].value_counts())
df.head(40)

In [ ]:
import pandas as pd

def parse_sample(fmt, smp):
    return dict(zip(fmt.split(":"), smp.split(":")))

recs = []
for _, r in df.iterrows():
    s = parse_sample(r["format"], r["sample"])
    recs.append({
        "gene": r["gene"],
        "chrom": r["chrom"],
        "pos": r["pos"],
        "rsid": None if r["id"] == "." else r["id"],
        "ref": r["ref"],
        "alt": r["alt"],
        "filter": r["filter"],
        "gt": s.get("GT"),
        "dp": int(s["DP"]) if s.get("DP", ".").isdigit() else None,
        "gq": int(s["GQ"]) if s.get("GQ", ".").isdigit() else None,
        "pid": s.get("PID"),
        "qual": float(r["qual"]),
    })

v = pd.DataFrame(recs)
v["novel"] = v["rsid"].isna()
v["passing"] = v["filter"] == "PASS"
v["indel"] = (v["ref"].str.len() != 1) | (v["alt"].str.len() != 1)

print(v.groupby("gene")[["passing", "novel", "indel"]].sum())
print()
print(v[v["gene"] == "BUB1B"].sort_values("pos").to_string())

In [ ]:
import requests, time, json
import pandas as pd

SERVER = "https://rest.ensembl.org"
ENDPOINT = "/vep/human/region"

v["vep_key"] = v["chrom"] + " " + v["pos"].astype(str) + " . " + v["ref"] + " " + v["alt"] + " . . ."

def vep_batch(variant_strings):
    r = requests.post(
        SERVER + ENDPOINT,
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        params={"canonical": 1, "hgvs": 1, "numbers": 1, "af": 1,
                "af_gnomade": 1, "af_gnomadg": 1, "SpliceRegion": 1},
        data=json.dumps({"variants": variant_strings}),
        timeout=120,
    )
    r.raise_for_status()
    return r.json()

keys = v["vep_key"].tolist()
raw = []
for i in range(0, len(keys), 150):
    raw.extend(vep_batch(keys[i:i + 150]))
    time.sleep(1)

def max_gnomad(rec):
    best = None
    for cv in rec.get("colocated_variants", []):
        for k, val in (cv.get("frequencies") or {}).get(rec.get("allele_string", "").split("/")[-1], {}).items():
            if "gnomad" in k.lower() and isinstance(val, (int, float)):
                best = val if best is None else max(best, val)
    return best

rows = []
for rec in raw:
    tcs = [t for t in rec.get("transcript_consequences", [])
           if t.get("gene_symbol") in ("BUB1B", "CEP57", "TRIP13")]
    canon = next((t for t in tcs if t.get("canonical")), tcs[0] if tcs else {})
    rows.append({
        "input": rec.get("input"),
        "symbol": canon.get("gene_symbol"),
        "consequence": rec.get("most_severe_consequence"),
        "exon": canon.get("exon"),
        "intron": canon.get("intron"),
        "hgvsc": canon.get("hgvsc"),
        "hgvsp": canon.get("hgvsp"),
        "sift": canon.get("sift_prediction"),
        "polyphen": canon.get("polyphen_prediction"),
        "gnomad_af": max_gnomad(rec),
    })

ann = pd.DataFrame(rows)
ann["pos"] = ann["input"].str.split().str[1].astype(int)
m = v.merge(ann.drop(columns=["input"]), on="pos", how="left")

GENE_BODY = {"BUB1B": (40160984, 40221137),
             "CEP57": (95789965, 95837070),
             "TRIP13": (892849, 919357)}
m["in_gene_body"] = [
    GENE_BODY[g][0] <= p <= GENE_BODY[g][1] for g, p in zip(m["gene"], m["pos"])
]

SEVERE = {"stop_gained", "frameshift_variant", "splice_acceptor_variant",
          "splice_donor_variant", "start_lost", "stop_lost",
          "missense_variant", "splice_region_variant", "inframe_deletion",
          "inframe_insertion"}

print(m["consequence"].value_counts(), "\n")
print(m[m["consequence"].isin(SEVERE) | m["novel"]]
      [["gene", "pos", "rsid", "gt", "consequence", "exon", "intron",
        "hgvsc", "hgvsp", "sift", "polyphen", "gnomad_af", "in_gene_body", "filter"]]
      .sort_values(["gene", "pos"]).to_string())

In [ ]:
import pandas as pd

PROBAND = "PROBAND01"

rows = [
    dict(proband_id=PROBAND,
         chrom_1="chr15", pos_1=40209701, ref_1="T", alt_1="G",
         chrom_2="chr15", pos_2=40220612, ref_2="T", alt_2="G",
         epcr=0.95, finding_type="primary",
         notes="BUB1B compound het: p.Leu737Ter (NMD-predicted null) + p.Asn1002Lys (kinase domain, SIFT deleterious, PolyPhen probably damaging, absent gnomAD)"),
    dict(proband_id=PROBAND,
         chrom_1="chr15", pos_1=40209701, ref_1="T", alt_1="G",
         chrom_2="", pos_2="", ref_2="", alt_2="",
         epcr=0.70, finding_type="primary",
         notes="Nonsense allele alone"),
    dict(proband_id=PROBAND,
         chrom_1="chr15", pos_1=40220612, ref_1="T", alt_1="G",
         chrom_2="", pos_2="", ref_2="", alt_2="",
         epcr=0.60, finding_type="primary",
         notes="Missense allele alone"),
    dict(proband_id=PROBAND,
         chrom_1="chr15", pos_1=40209701, ref_1="T", alt_1="G",
         chrom_2="chr15", pos_2=40216470, ref_2="A", alt_2="G",
         epcr=0.30, finding_type="primary",
         notes="Alternative second allele: deep intronic c.2679-1026A>G, absent gnomAD, untested for pseudoexon activation"),
    dict(proband_id=PROBAND,
         chrom_1="chr15", pos_1=40216470, ref_1="A", alt_1="G",
         chrom_2="", pos_2="", ref_2="", alt_2="",
         epcr=0.15, finding_type="secondary",
         notes="Deep intronic BUB1B variant, absent gnomAD, flagged for splicing follow-up"),
]

sub = pd.DataFrame(rows)
sub.to_csv("nandini-gantayat_bub1b-panel-vep.csv", index=False)
sub